# Challenge 10 — Feature Engineering Challenge

**Nombre:** Miriam Ivonne Castañeda Chavarría
**Clase:** Feature Engineering  

## Objetivo

Intentar mejorar un modelo de **LinearRegression** modificando solamente la representación de las variables.

**Dataset:** `viviendas_regresion.csv`

La idea es comparar un modelo base contra otro con nuevas features.


## 1. Cargar y revisar los datos

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/ivonnecastaneda/Downloads/viviendas_regresion.csv")

print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nValores faltantes:")
print(df.isna().sum())

df.head()


Dimensiones: (500, 7)

Columnas:
['metros', 'habitaciones', 'banios', 'antiguedad', 'distancia_centro_km', 'vista_panoramica_score', 'precio']

Valores faltantes:
metros                    0
habitaciones              0
banios                    0
antiguedad                0
distancia_centro_km       0
vista_panoramica_score    0
precio                    0
dtype: int64


,metros,habitaciones,banios,antiguedad,distancia_centro_km,vista_panoramica_score,precio
0,46.0,4,2,41,1.57,4.41,640982.0
1,94.3,2,3,40,18.83,8.93,1074328.0
2,182.9,2,2,15,14.00,0.36,2541204.0
3,265.8,2,2,39,1.10,8.55,3908903.0
4,60.4,5,2,40,7.86,1.63,944055.0


No hay valores faltantes en este archivo. Tampoco hay una columna de fecha.

Por eso no se puede crear una variable de calendario sin inventar información. Para la parte temporal se conserva `antiguedad`, que representa años y está directamente relacionada con el paso del tiempo de la vivienda.


## 2. Modelo base

In [2]:
X = df[["metros", "habitaciones", "banios","antiguedad", "distancia_centro_km"]]
y = df["precio"]
X.head()

,metros,habitaciones,banios,antiguedad,distancia_centro_km
0,46.0,4,2,41,1.57
1,94.3,2,3,40,18.83
2,182.9,2,2,15,14.00
3,265.8,2,2,39,1.10
4,60.4,5,2,40,7.86


El modelo base usa variables básicas de la vivienda.

El algoritmo será **LinearRegression** y se mantendrá igual durante todo el ejercicio.


## 3. Train / Test

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (400, 5)
Test: (100, 5)


La división se hace antes de entrenar el preprocessing. Esto evita que el test participe en el aprendizaje de parámetros.


## 4. Pipeline del modelo base

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

modelo_base = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),("model", LinearRegression())])
modelo_base.fit(X_train, y_train)
pred_base = modelo_base.predict(X_test)

## 5. Métricas del modelo base

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_base = mean_absolute_error(y_test, pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, pred_base))
r2_base = r2_score(y_test, pred_base)
print("MAE base:", mae_base)
print("RMSE base:", rmse_base)
print("R2 base:", r2_base)

MAE base: 109403.53040903086
RMSE base: 149698.52389205224
R2 base: 0.9840283396443262


## 6. Proponer nuevas features

In [8]:
df_fe = df.copy()

# 1. Ratio
df_fe["metros_por_habitacion"] = (df_fe["metros"] / df_fe["habitaciones"].replace(0, np.nan))

# 2. Variable relacionada con el tiempo
df_fe["antiguedad_tiempo"] = df_fe["antiguedad"]

# 3. Transformación numérica
df_fe["log_metros"] = np.log1p(df_fe["metros"])

# Otra transformación
df_fe["antiguedad_cuadrado"] = df_fe["antiguedad"] ** 2

# Otra feature útil
df_fe["distancia_inversa"] = (1 / (1 + df_fe["distancia_centro_km"]))

# 4. Variable categórica codificada
df_fe["distancia_categoria"] = pd.cut(df_fe["distancia_centro_km"],bins=[-np.inf, 5, 10, np.inf],labels=["Cerca", "Media", "Lejos"])
df_fe.head()

,metros,habitaciones,banios,antiguedad,distancia_centro_km,vista_panoramica_score,precio,metros_por_habitacion,antiguedad_tiempo,log_metros,antiguedad_cuadrado,distancia_inversa,distancia_categoria
0,46.0,4,2,41,1.57,4.41,640982.0,11.50,41,3.850148,1681,0.389105,Cerca
1,94.3,2,3,40,18.83,8.93,1074328.0,47.15,40,4.557030,1600,0.050429,Lejos
2,182.9,2,2,15,14.00,0.36,2541204.0,91.45,15,5.214392,225,0.066667,Lejos
3,265.8,2,2,39,1.10,8.55,3908903.0,132.90,39,5.586499,1521,0.476190,Cerca
4,60.4,5,2,40,7.86,1.63,944055.0,12.08,40,4.117410,1600,0.112867,Media


### ¿Qué agregamos?

- `metros_por_habitacion`: ratio.
- `antiguedad_tiempo`: variable relacionada con el tiempo.
- `log_metros`: transformación numérica.
- `antiguedad_cuadrado`: permite representar una relación no lineal.
- `distancia_inversa`: transforma la distancia.
- `distancia_categoria`: convierte la distancia en una variable categórica.


## 7. Revisar divisiones, infinitos y faltantes

In [9]:
print("Infinitos:")
print(np.isinf(df_fe.select_dtypes(include=np.number)).sum())

print("\nFaltantes:")
print(df_fe.isna().sum())


Infinitos:
metros                    0
habitaciones              0
banios                    0
antiguedad                0
distancia_centro_km       0
vista_panoramica_score    0
precio                    0
metros_por_habitacion     0
antiguedad_tiempo         0
log_metros                0
antiguedad_cuadrado       0
distancia_inversa         0
dtype: int64

Faltantes:
metros                    0
habitaciones              0
banios                    0
antiguedad                0
distancia_centro_km       0
vista_panoramica_score    0
precio                    0
metros_por_habitacion     0
antiguedad_tiempo         0
log_metros                0
antiguedad_cuadrado       0
distancia_inversa         0
distancia_categoria       0
dtype: int64


La división de `metros / habitaciones` se protegió contra división entre cero. Los posibles faltantes resultantes serán manejados dentro del Pipeline mediante la mediana.


## 8. Train / Test para las nuevas features

In [10]:
features = ["metros", "habitaciones", "banios", "antiguedad","distancia_centro_km", "vista_panoramica_score","metros_por_habitacion", "distancia_inversa",
    "log_metros", "antiguedad_cuadrado","antiguedad_tiempo", "distancia_categoria"]
X_fe = df_fe[features]
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe,y,test_size=0.20,random_state=42)

print(X_train_fe.shape)
print(X_test_fe.shape)

(400, 12)
(100, 12)


## 9. ColumnTransformer + Pipeline

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numericas = ["metros", "habitaciones", "banios", "antiguedad","distancia_centro_km", "vista_panoramica_score",
    "metros_por_habitacion", "distancia_inversa","log_metros", "antiguedad_cuadrado","antiguedad_tiempo"]

categoricas = ["distancia_categoria"]
preprocesamiento = ColumnTransformer([("numericas", Pipeline([("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())]), numericas),("categoricas", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),("onehot", OneHotEncoder(handle_unknown="ignore"))]), categoricas)])
modelo_mejorado = Pipeline([("preprocesamiento", preprocesamiento),("model", LinearRegression())])
modelo_mejorado.fit(X_train_fe, y_train_fe)
pred_mejorado = modelo_mejorado.predict(X_test_fe)

## 10. Métricas del modelo mejorado

In [13]:
mae_mejorado = mean_absolute_error(y_test_fe, pred_mejorado)
rmse_mejorado = np.sqrt(mean_squared_error(y_test_fe, pred_mejorado))
r2_mejorado = r2_score(y_test_fe, pred_mejorado)

print("MAE mejorado:", mae_mejorado)
print("RMSE mejorado:", rmse_mejorado)
print("R2 mejorado:", r2_mejorado)

MAE mejorado: 115715.84713178773
RMSE mejorado: 154385.1103765979
R2 mejorado: 0.9830126413735973


## 11. Comparación

In [14]:
comparacion = pd.DataFrame({"Modelo": ["Base", "Feature Engineering"],"MAE": [mae_base, mae_mejorado],
    "RMSE": [rmse_base, rmse_mejorado],"R2": [r2_base, r2_mejorado]})
comparacion

,Modelo,MAE,RMSE,R2
0,Base,109403.530409,149698.523892,0.984028
1,Feature Engineering,115715.847132,154385.110377,0.983013


## 12. ¿Mejoró el modelo?

In [15]:
print("Cambio en MAE:", mae_mejorado - mae_base)
print("Cambio en RMSE:", rmse_mejorado - rmse_base)
print("Cambio en R2:", r2_mejorado - r2_base)

if mae_mejorado < mae_base:
    print("\nEl Feature Engineering mejoró el MAE.")
else:
    print("\nEl Feature Engineering no mejoró el MAE.")

Cambio en MAE: 6312.316722756863
Cambio en RMSE: 4686.586484545667
Cambio en R2: -0.001015698270728893

El Feature Engineering no mejoró el MAE.


## Conclusión

Se mantuvo el mismo algoritmo, `LinearRegression`, tanto en el modelo base como en el modelo mejorado.  
La principal diferencia fue la representación de las variables de entrada.  
Se agregaron un ratio, transformaciones numéricas, una variable relacionada con el tiempo y una variable categórica codificada.  
También se utilizó `ColumnTransformer` para tratar por separado variables numéricas y categóricas.  
La comparación final de MAE, RMSE y R² permite comprobar si las nuevas features realmente aportaron información.  
Si una feature empeora el resultado, no necesariamente debe conservarse en una versión final.
